# Trabalho 1 — Modelagem com Árvore de Decisão e Ensembles

Nome 1: 

Nome 2:

Este exercício utiliza a base **Bank Marketing (UCI)**, composta por dados de campanhas de telemarketing de um banco português realizadas entre 2008 e 2010. Cada registro descreve características do cliente, do contato realizado e do contexto da campanha, e o objetivo é prever se o cliente contratará um depósito a prazo (y ∈ {yes, no}).

## Parte 1 - Análise Exploratória de Dados

Antes de qualquer modelagem, é necessário construir uma compreensão sólida da base de dados em estudo. Nesta etapa, precisamos: verificar a estrutura da base (dimensões, tipos), a distribuição do alvo e o desbalanceamento, mapear a qualidade dos dados, avaliar a cardinalidade das variáveis categóricas e o comportamento das numéricas, inspecionar a dimensão temporal e identificar riscos de vazamento de informação (data leakage).

Para implementar esta análise, você deverá responder as perguntas abaixo. Além da resposta por extenso, você deve apresentar o código que usou para encontrar a resposta. Sempre que possível, adicione comentários nas linhas de código para ajudar o professor na correção

In [21]:
import pandas as pd
import numpy as np

# O CSV da UCI usa ponto e vírgula como separador
df = pd.read_csv("bank-additional-full.csv", sep=";")

# Mostrar as primeiras linhas
print(df.head())

   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed

### 1.1 Quantas linhas e colunas há no dataset? Alguma coluna é constante (valor único)?

Resposta:

In [22]:
num_linhas, num_colunas = df.shape
print(f"O dataset possui {num_linhas} linhas e {num_colunas} colunas.")


O dataset possui 41188 linhas e 21 colunas.


In [23]:
encontrou_constante = False

for coluna in df.columns:
    # Verifica se a coluna tem apenas 1 valor único
    if df[coluna].nunique() == 1:
        encontrou_constante = True
        valor_unico = df[coluna].unique()[0]
        print(f"A coluna '{coluna}' é constante com o valor: {valor_unico}")

if not encontrou_constante:
    print("Nenhuma coluna com valor constante foi encontrada no dataset.")


Nenhuma coluna com valor constante foi encontrada no dataset.


### 1.2 Quais são os tipos de dados por coluna? Há algo que deveria ser numérico e veio como texto?

Resposta:

In [24]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [25]:
colunas_resumidas = {
    'age': 'idade',
    'job': 'profissao',
    'marital': 'est_civil',
    'education': 'escolaridade',
    'default': 'inadimplencia',
    'housing': 'cred_habitacao',
    'loan': 'cred_pessoal',
    'contact': 'tipo_contato',
    'month': 'mes',
    'day_of_week': 'dia_semana_ultimo_contato',
    'duration': 'duracao_ultimo_contato_seg',
    'campaign': 'num_contatos_campanha',
    'pdays': 'dias_cont_anterior',
    'previous': 'contatos_previos',
    'poutcome': 'resultado_anterior',
    'emp.var.rate': 'var_emprego',
    'cons.price.idx': 'ipc', 
    'cons.conf.idx': 'icc', 
    'euribor3m': 'euribor_3m', 
    'nr.employed': 'num_empregados',
    'y': 'y'
}

# Renomeando as colunas
df = df.rename(columns=colunas_resumidas)


print(df.columns)

Index(['idade', 'profissao', 'est_civil', 'escolaridade', 'inadimplencia',
       'cred_habitacao', 'cred_pessoal', 'tipo_contato', 'mes',
       'dia_semana_ultimo_contato', 'duracao_ultimo_contato_seg',
       'num_contatos_campanha', 'dias_cont_anterior', 'contatos_previos',
       'resultado_anterior', 'var_emprego', 'ipc', 'icc', 'euribor_3m',
       'num_empregados', 'y'],
      dtype='object')


In [26]:
print(df.dtypes)

idade                           int64
profissao                      object
est_civil                      object
escolaridade                   object
inadimplencia                  object
cred_habitacao                 object
cred_pessoal                   object
tipo_contato                   object
mes                            object
dia_semana_ultimo_contato      object
duracao_ultimo_contato_seg      int64
num_contatos_campanha           int64
dias_cont_anterior              int64
contatos_previos                int64
resultado_anterior             object
var_emprego                   float64
ipc                           float64
icc                           float64
euribor_3m                    float64
num_empregados                float64
y                              object
dtype: object


In [27]:
# 1. Conversão da variável alvo 'y' (a mais importante)
# Modelos de machine learning precisam de um alvo numérico.
# Mapeamos 'no' para 0 e 'yes' para 1
df['y'] = df['y'].map({'no': 0, 'yes': 1}).astype('int')


# 2. Otimização das variáveis categóricas (de 'object' para 'category')
# O tipo 'category' é mais eficiente em memória e desempenho para colunas
# com um número limitado de valores repetidos.

# Lista das colunas que são categóricas (todas que eram 'object', exceto 'y')
colunas_para_converter = [
    'profissao', 'est_civil', 'escolaridade', 'inadimplencia',
    'cred_habitacao', 'cred_pessoal', 'tipo_contato', 'mes',
    'dia_semana_ultimo_contato', 'resultado_anterior'
]

# Loop para aplicar a conversão
for coluna in colunas_para_converter:
    df[coluna] = df[coluna].astype('category')


# 3. Verificação final dos tipos de dados
#print("--- Tipos de dados DEPOIS da otimização ---")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   idade                       41188 non-null  int64   
 1   profissao                   41188 non-null  category
 2   est_civil                   41188 non-null  category
 3   escolaridade                41188 non-null  category
 4   inadimplencia               41188 non-null  category
 5   cred_habitacao              41188 non-null  category
 6   cred_pessoal                41188 non-null  category
 7   tipo_contato                41188 non-null  category
 8   mes                         41188 non-null  category
 9   dia_semana_ultimo_contato   41188 non-null  category
 10  duracao_ultimo_contato_seg  41188 non-null  int64   
 11  num_contatos_campanha       41188 non-null  int64   
 12  dias_cont_anterior          41188 non-null  int64   
 13  contatos_previos

### 1.3 Em relação à variável alvo (target), qual a distribuição de classes (yes vs no) em contagem e proporção?

Resposta:

In [51]:
# 1. Calcular a contagem de cada classe (0 e 1)
contagem_classes = df['y'].value_counts()

# 2. Calcular a proporção (porcentagem) de cada classe
proporcao_classes = df['y'].value_counts(normalize=True) * 100

# 3. Criar um DataFrame para exibir os resultados de forma organizada
distribuicao_df = pd.DataFrame({
    'Contagem': contagem_classes,
    'Proporção (%)': proporcao_classes.round(2) # Arredondando para 2 casas decimais
})

# 4. Mapear os índices para tornar a saída mais clara
distribuicao_df.index = distribuicao_df.index.map({0: 'Não Aderiu (0)', 1: 'Aderiu (1)'})

# Exibir o resultado
print(distribuicao_df)

# Adicionar uma análise textual sobre o resultado
print("\n--- Análise ---")
if proporcao_classes.min() < 20: # Se a menor classe for menos de 20%
    print("O conjunto de dados é bastante desbalanceado.")
    print("A classe majoritária (Não Aderiu) é muito mais frequente que a classe minoritária (Aderiu).")
    print("Isso é um ponto de atenção para a etapa de modelagem, pois o modelo pode tender a 'apostar' sempre na classe majoritária.")
else:
    print("O conjunto de dados possui um balanceamento razoável entre as classes.")

                Contagem  Proporção (%)
y                                      
Não Aderiu (0)     36548          88.73
Aderiu (1)          4640          11.27

--- Análise ---
O conjunto de dados é bastante desbalanceado.
A classe majoritária (Não Aderiu) é muito mais frequente que a classe minoritária (Aderiu).
Isso é um ponto de atenção para a etapa de modelagem, pois o modelo pode tender a 'apostar' sempre na classe majoritária.


### 1.4 Essa proporção sugere desbalanceamento? Se sim, quais métricas de avaliação de desempenho serão mais adequadas (e por quê)?

Resposta:

In [29]:
#Insira seu código aqui

### 1.5 Qual a cardinalidade (número de categorias distintas) de cada coluna categórica? Liste da maior para a menor.

Resposta:

In [30]:
#Insira seu código aqui

### 1.6 Mostre estatísticas descritivas (min/mediana/média/p95/p99) para as variáveis numéricas.

Resposta:

In [31]:
#Insira seu código aqui

### 1.7 Há outliers evidentes em campaign, age, duration? Como tratá-los (ou não) e por quê?

Resposta:

In [32]:
#Insira seu código aqui

### 1.8 Sobre as variáveis previous e poutcome, o que elas representam? Qual a relação delas com a taxa de yes no target?

Resposta:

In [33]:
#Insira seu código aqui

### 1.9 Qual a distribuição de registros por 'month'? Há sazonalidade? A taxa de 'yes' no target varia por 'month'? Mostre graficamente

Resposta:

In [34]:
#Insira seu código aqui

### 1.10 Qual a correlação/taxa de 'yes' por faixa de 'duration'? Considerando o resultado obtido e a natureza da variável 'duration', ela deve entrar no treinamento dos modelos?

Resposta:

In [35]:
#Insira seu código aqui

### 1.11 Existem colunas que incorporam informação “do futuro” ou pós-contato? Quais e por quê?

Resposta:

In [36]:
#Insira seu código aqui

## Parte 2 - Pré-processamento

Nesta Parte 2 (Pré-processamento) você vai transformar o "Bank Marketing" de uma base de dados bruta para uma pronta para modelagem. Isso inclui: remover variáveis inadequadas ao treino, decidir o split treino/teste e a forma de validação, tratar 'unknown', realizar balanceamento, etc. Ao final, você terá X_train, y_train, X_test, y_test e um preprocessador reutilizável para plugar diretamente nos modelos da Parte 3.

### 2.1 Qual é a lista final de colunas de entrada? Alguma deve ser removida? Justifique e remova-as caso necessário

Resposta:

In [37]:
#Insira seu código aqui

### 2.2 Existem linhas duplicadas? Se sim, o que fazer com elas?

Resposta:

In [38]:
#Insira seu código aqui

### 2.3 Em quais colunas categóricas aparece a categoria unknown? Qual a proporção por coluna? Você trataria unknown como categoria válida, imputaria ou agregaria? Justifique para as duas colunas com maior quantidade de unknown.

Resposta:

In [39]:
#Insira seu código aqui

### 2.4 O que a variável pdays representa? Como ela precisa ser tratada no pré-processamento?

Resposta:

In [40]:
#Insira seu código aqui

### 2.5 Existem outliers que precisam ser removidos? Justifique

Resposta:

In [41]:
#Insira seu código aqui

### 2.6 Alguma variável precisa ser normalizada ou padronizada? Justifique

Resposta:

In [42]:
#Insira seu código aqui

### 2.7 Faça a divisão treino/teste e construa X_train, X_test, y_train, y_test. (Cuidado com a divisão em treino/teste com séries temporais!)

Resposta:

In [43]:
#Insira seu código aqui

### 2.8 Codifique variáveis categóricas (Sugestão: use "ColumnTransformer")

Resposta:

In [44]:
#Insira seu código aqui

### 2.9 Se achar necessário, adicione aqui alguma atividade de pré-processamento para melhorar o desempenho dos modelos a serem treinados

Resposta:

In [45]:
#Insira seu código aqui

## Parte 3 - Treinando e Avaliando os Modelos

Nesta etapa você treinará os modelos usando exclusivamente o conjunto de treino. O foco aqui é treinar modelos das técnicas vistas em sala de aula (Árvore de Decisão, Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost) para predizer a variável target. Dpoies, usando o conjunto de teste, deve-se avaliar o desempenho de cada modelo.

O que será avaliado?
 - Se todos os modelos foram treinados corretamente
 - Se as métricas de desempenho (Matriz de confusão, Acurácia, Precisão, Recall, F1-Score, ROC AUC, PR AUC, etc.) foram calculadas
 - Se uma grid search (pode ser algo curto, 2 a 10 valores, por exemplo) foi feita para pelo menos um dos hiperparâmetros de cada modelo
 - Se o código está bem comentado, explicando o seu funcionamento
 
 OBS: Pontuação extra será dada para os alunos que realizarem testes ou implementarem estratégias além da minimamente exigida para melhorar o desempenho dos modelos. Figuras e gráficos ilustrativos, para ajudar na interpretação de modelos e resultados, também marcam pontos extras


In [46]:
#Insira seu código aqui

## Parte 4 - Conclusões

O objetivo desta etapa final é interpretar resultados, refletir sobre o pré-processamento e consolidar recomendações.

### 4.1 Quais métricas de desempenho são mais relevantes, neste caso, para comparar os diferentes modelos?

Resposta:

In [47]:
#Insira seu código aqui

### 4.2 Qual modelo obteve os melhores AP (PR AUC) e ROC AUC no teste?

Resposta:

In [48]:
#Insira seu código aqui

### 4.3 Dentre todos os modelos treinados, indique o que teve o melhor desempenho (descreva o tipo e hiperparâmetros). Justifique a sua escolha

Resposta:

In [49]:
#Insira seu código aqui

### 4.4 Na sua opinião, qual etapa, de todas as executadas para a obtenção dos resultados, foi mais decisiva e qual foi a mais trabalhosa? O que você faria diferente em um próximo projeto?

Resposta:

In [50]:
#Insira seu código aqui